# 8. Imbalance strategies

**Question: does rebalancing the training data beat simply moving the threshold?**

This is the notebook most likely to overturn a habit.

The standard response to 1.7% positives is to rebalance: undersample the majority,
synthesise minority points with SMOTE, or weight the classes. All three change what the
model learns. But there is a fourth option that changes nothing about training at all —
leave the data alone and move the decision threshold.

Since every model in this study already gets a cost-optimal threshold, the fair question
is not "does rebalancing beat 0.5" — it obviously does — but **"does rebalancing beat
threshold tuning?"**

One correctness note: resampling happens strictly inside a fit-only
`imbalanced-learn` pipeline. Validation, calibration, threshold and test data are never
resampled. Synthesising SMOTE points before splitting is the classic way to publish a
result that evaporates in production.

> **The objective.** Every number in this notebook is judged against
> `J = 10·FP + 500·FN`. A false positive is an unnecessary inspection; a false
> negative is a truck that fails in service. Missing one failure costs as much
> as fifty needless inspections, and that ratio is what makes the modelling
> choices here matter.

In [ ]:
from pathlib import Path

import pandas as pd

from scania_aps.data import TEST_FILENAME, TRAIN_FILENAME, read_raw_csv
from scania_aps.plotting import apply_house_style

ROOT = Path.cwd().resolve()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
TRAIN = ROOT / "data" / "raw" / TRAIN_FILENAME
TEST = ROOT / "data" / "raw" / TEST_FILENAME
ARTIFACTS = ROOT / "artifacts"
assert TRAIN.exists() and TEST.exists(), "Run: poetry run scania-aps download"

apply_house_style()

train = read_raw_csv(TRAIN)
test = read_raw_csv(TEST)

pd.DataFrame(
    {
        "trucks": [len(train.y), len(test.y)],
        "features": [train.X.shape[1], test.X.shape[1]],
        "failures": [int(train.y.sum()), int(test.y.sum())],
        "failure_rate": [train.y.mean(), test.y.mean()],
    },
    index=["training set", "official test set"],
)

In [ ]:
from scania_aps.studies import run_imbalance_study

imbalance = run_imbalance_study(TRAIN, TEST, ARTIFACTS)
imbalance

In [ ]:
from scania_aps.plotting import emphasis_bars

label_column = "strategy" if "strategy" in imbalance.columns else imbalance.columns[0]

fig, ax = emphasis_bars(
    [str(v) for v in imbalance[label_column]],
    [float(v) for v in imbalance["total_cost"].to_numpy()],
    title="Maintenance cost by imbalance strategy",
    subtitle="Official test set, each strategy at its own cost-optimal threshold.",
    xlabel="Maintenance cost  (10·FP + 500·FN)",
)

## The error trade behind the cost

A single cost number hides *how* each strategy got there. Two strategies can cost the
same while making very different mistakes, and for a maintenance planner the mix
matters: false negatives are trucks that break down, false positives are workshop
hours.

In [ ]:
errors = imbalance[
    [label_column, "false_negatives", "false_positives", "recall", "precision", "total_cost"]
].copy()
errors["cost_from_fn"] = errors["false_negatives"] * 500
errors["cost_from_fp"] = errors["false_positives"] * 10
errors["fn_share_of_cost"] = errors["cost_from_fn"] / errors["total_cost"]
errors.sort_values("total_cost")

### Reading the result

Look at `fn_share_of_cost`. Under this cost ratio almost every sensible operating point
is dominated by the false-negative term — which is the arithmetic telling you that the
job is recall, bought as cheaply as possible in inspections.

If the resampling strategies land close to the no-correction baseline, the honest
conclusion is that **thresholding already did the work**, and SMOTE's extra complexity
and runtime bought nothing. That is a genuinely useful negative result: it is much
cheaper to move a threshold than to synthesise a training set.

**Next:** [09 — probability calibration](09_probability_calibration.ipynb), which asks
whether the scores being thresholded mean anything.